# Network Anomaly Detection

Notebook pedagogique pour analyser des donnees reseau et detecter des activites anormales.

## Introduction

Objectif: construire un pipeline simple et credibile de cybersecurite data-driven.

Etapes:
- Chargement des donnees
- Nettoyage
- Analyse exploratoire
- Preparation des variables
- Modelisation
- Resultats
- Conclusion

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

# Resolve project root whether notebook is launched from root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from preprocessing import (
    build_preprocessor,
    clean_network_data,
    load_dataset,
    missing_values_summary,
    split_features_target,
)
from modeling import (
    evaluate_supervised_model,
    evaluate_unsupervised_predictions,
    predict_anomaly_with_isolation_forest,
    select_modeling_strategy,
    train_isolation_forest,
    train_supervised_model,
)
from utils import ensure_directories, save_figure, save_text_report

FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
REPORTS_DIR = PROJECT_ROOT / "results" / "reports"
ensure_directories([FIGURES_DIR, REPORTS_DIR])

plt.style.use("ggplot")

## Chargement des donnees

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "network_data.csv"
df_raw = load_dataset(DATA_PATH)

print(f"Shape brute: {df_raw.shape}")
df_raw.head()

## Nettoyage

In [ ]:
df_clean = clean_network_data(df_raw, target_column="label")

print(f"Shape apres nettoyage: {df_clean.shape}")
print("\nValeurs manquantes:")
display(missing_values_summary(df_clean))

df_clean.head()

## Analyse exploratoire

In [ ]:
print("Statistiques descriptives (variables numeriques):")
display(df_clean.describe(include=["number"]).T)

if "label" in df_clean.columns:
    class_counts = df_clean["label"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    class_counts.plot(kind="bar", ax=ax, color=["#4CAF50", "#D32F2F"])
    ax.set_title("Distribution des classes")
    ax.set_xlabel("Label")
    ax.set_ylabel("Nombre de connexions")
    ax.set_xticklabels(["Normal (0)", "Anomaly (1)"], rotation=0)
    save_figure(fig, FIGURES_DIR / "class_distribution.png")
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_clean["src_bytes"].plot(kind="hist", bins=20, ax=axes[0], color="#1976D2")
axes[0].set_title("Distribution de src_bytes")
axes[0].set_xlabel("src_bytes")

df_clean["dst_bytes"].plot(kind="hist", bins=20, ax=axes[1], color="#00897B")
axes[1].set_title("Distribution de dst_bytes")
axes[1].set_xlabel("dst_bytes")

save_figure(fig, FIGURES_DIR / "traffic_histograms.png")
plt.show()

if "label" in df_clean.columns:
    fig, ax = plt.subplots(figsize=(7, 5))
    scatter = ax.scatter(
        df_clean["connection_rate"],
        df_clean["failed_logins"],
        c=df_clean["label"],
        cmap="coolwarm",
        alpha=0.8,
    )
    ax.set_title("Connection rate vs failed logins")
    ax.set_xlabel("connection_rate")
    ax.set_ylabel("failed_logins")
    legend = ax.legend(*scatter.legend_elements(), title="Label")
    ax.add_artist(legend)
    save_figure(fig, FIGURES_DIR / "connection_vs_logins.png")
    plt.show()

## Preparation des donnees

In [ ]:
X, y = split_features_target(df_clean, target_column="label")
preprocessor = build_preprocessor(X)

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y,
    )
else:
    X_train, X_test = train_test_split(X, test_size=0.3, random_state=42)
    y_train, y_test = None, None

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")

## Modelisation

In [ ]:
strategy = select_modeling_strategy(y)
print(f"Strategie choisie: {strategy}")

if strategy == "supervised":
    model = train_supervised_model(X_train, y_train, preprocessor)
    evaluation = evaluate_supervised_model(model, X_test, y_test)
    y_pred = model.predict(X_test)
else:
    model = train_isolation_forest(X_train, preprocessor, contamination=0.2)
    y_pred = predict_anomaly_with_isolation_forest(model, X_test)
    evaluation = evaluate_unsupervised_predictions(y_test, y_pred)

evaluation

## Resultats

In [ ]:
if "classification_report" in evaluation:
    report_df = pd.DataFrame(evaluation["classification_report"]).T
    display(report_df)

if "confusion_matrix" in evaluation:
    cm = evaluation["confusion_matrix"]
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title("Matrice de confusion")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax)
    save_figure(fig, FIGURES_DIR / "confusion_matrix.png")
    plt.show()

summary_lines = [
    f"Modeling strategy: {strategy}",
    f"Accuracy: {evaluation.get('accuracy', 'n/a'):.4f}" if isinstance(evaluation.get('accuracy', None), float) else f"Accuracy: {evaluation.get('accuracy', 'n/a')}",
    f"Precision: {evaluation.get('precision', 'n/a'):.4f}" if isinstance(evaluation.get('precision', None), float) else f"Precision: {evaluation.get('precision', 'n/a')}",
    f"Recall: {evaluation.get('recall', 'n/a'):.4f}" if isinstance(evaluation.get('recall', None), float) else f"Recall: {evaluation.get('recall', 'n/a')}",
    f"F1-score: {evaluation.get('f1_score', 'n/a'):.4f}" if isinstance(evaluation.get('f1_score', None), float) else f"F1-score: {evaluation.get('f1_score', 'n/a')}",
]

if "roc_auc" in evaluation:
    summary_lines.append(f"ROC-AUC: {evaluation['roc_auc']:.4f}")

save_text_report("\n".join(summary_lines), REPORTS_DIR / "model_summary.txt")
print("Rapport ecrit dans results/reports/model_summary.txt")

results_preview = X_test.copy()
results_preview["predicted_label"] = y_pred
if y_test is not None:
    results_preview["true_label"] = y_test.values

results_preview.head(10)

## Conclusion

Ce notebook montre un workflow complet et interpretable pour la detection d'anomalies reseau.

Points importants:
- Les connexions anormales presentent souvent des taux de connexion eleves et plus d'echecs d'authentification.
- Les performances de classification dependent fortement de la qualite des labels et du volume de donnees.
- Pour un usage reel, il faudrait enrichir le dataset (plus de trafic, plus de contextes, validation temporelle).